# Notebook 3 — Small-Step Operational Semantics

**Covers Chapter 2 §2.1–2.4.** This is where we make precise what "running a While program" actually *means*. We zoom in on:

- States σ — the formal model of "the values of all variables right now."
- The state update notation σ[x ↦ n].
- The function `A` — interprets arithmetic expressions in a state.
- The function `B` — interprets boolean expressions in a state.
- The small-step transition relation `⟨S, σ⟩ ⇒ ⟨S', σ'⟩`.
- The 6 inference rules: `:=`, `;` (two cases), `if` (two cases), `while` (two cases).

Plus all chapter examples 7–11 reproduced as live interpreter runs, and exercises 11–16. Exercise 14 (the Diophantine derivation) is the showpiece.

## What you'll be able to do by the end

- Apply the small-step rules by hand to any While program.
- Reproduce Example 7's formal trace from scratch (or fix it if I made a mistake).
- Produce the state-tracking table format the exam asks for.
- Argue why the encoding from Exercise 4 (no-if) behaves identically to the original `if`.

In [1]:
import sys
from pathlib import Path

# Make this work whether jupyter was started from notebooks/ or from the repo root.
for candidate in [Path.cwd(), Path.cwd() / 'notebooks', Path.cwd().parent / 'notebooks']:
    if (candidate / 'while_lang.py').exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    raise ImportError("could not find while_lang.py — start jupyter from the notebooks/ folder")

from while_lang import (
    parse, run, trace,
    A, B, step, step_iter,
    Config, Transition, StepBudgetExceeded,
    check_state, check_steps,
    state_to_str, cfg_to_str, stmt_to_str,
)

## §1. States — Definition 1

From the chapter:

> A **state** is a function σ : Vars → ℤ such that there are only finitely many variables `x ∈ Vars` with σ(x) ≠ 0. We use **St** for the set of all states.

Key bits:

- A state is a **function**, not a list. Its inputs are variable names; its outputs are integers.
- The constraint: only finitely many variables map to non-zero values. All others map to 0.
- We use Greek letters (σ, τ, ρ, ...) for state names so we don't confuse them with anything else.

**Why finitely-many-non-zero?** A program text mentions only finitely many variable names. Anything not mentioned has no specified value. The chapter's clean choice: assume those default to 0, so we can talk about states without worrying about "which variables are tracked."

### State description notation

The chapter writes a state with all-but-finitely-many zeros as:

$$\sigma = \{m \mapsto 10,\ n \mapsto 3\}$$

This means: σ(m) = 10, σ(n) = 3, σ(x) = 0 for every other variable x.

**Important** — two distinct rules apply to this notation:

1. In a *state description* (curly braces, no σ named), every variable mentioned must be **distinct**. `{x ↦ 1, x ↦ 2}` is not a valid state description.
2. In a *state update* (next subsection), variables can repeat — and the last update wins.

Our interpreter models a state as a Python dict, with `.get(name, 0)` for the default-zero behavior.

In [2]:
sigma = {'m': 10, 'n': 3}
print('σ(m) =', sigma.get('m', 0))            # 10
print('σ(n) =', sigma.get('n', 0))            # 3
print('σ(banana) =', sigma.get('banana', 0))  # 0 (the default-zero rule)

# Pretty-printed in chapter notation:
print('σ =', state_to_str(sigma))

σ(m) = 10
σ(n) = 3
σ(banana) = 0
σ = {m ↦ 10, n ↦ 3}


## §2. State updates — σ[x ↦ n]

The chapter defines:

$$(\sigma[x \mapsto n])(y) = \begin{cases} n & y = x \\ \sigma(y) & \text{otherwise} \end{cases}$$

**Read this carefully** — the update notation `σ[x ↦ n]` defines a *new state* (a new function), not a mutation of σ. The new state agrees with σ on every variable except x; on x, it gives n.

### Multiple updates

Repeated update notation is unwieldy:

$$(\sigma[d \mapsto 3])[r \mapsto 1]$$

The chapter introduces a shortcut: `σ[d ↦ 3, r ↦ 1]` means "first apply `d ↦ 3`, then `r ↦ 1`." The order matters when the same variable is updated twice — the *last* update wins:

$$\sigma[x \mapsto 1, x \mapsto 2] = \sigma[x \mapsto 2]$$

**Subtle point:** `σ[x ↦ σy]` updates x to whatever value σ assigns y. If σ doesn't mention y, then σy = 0, and we get `σ[x ↦ 0]`.

In [3]:
# Update operation in Python: {**sigma, x: n}
sigma = {'m': 10, 'n': 3}
tau = {**sigma, 'r': sigma.get('m', 0)}      # σ[r ↦ σm]
print('after σ[r ↦ σm]:', state_to_str(tau))

rho = {**tau, 'd': 0}                         # τ[d ↦ 0]
print('then τ[d ↦ 0]:  ', state_to_str(rho))

# Multi-update shortcut as a single dict-update step:
rho2 = {**sigma, 'r': sigma.get('m', 0), 'd': 0}
print('σ[r ↦ σm, d ↦ 0]:', state_to_str(rho2))
# rho == rho2 — the multi-update is just shorthand.

after σ[r ↦ σm]: {m ↦ 10, n ↦ 3, r ↦ 10}
then τ[d ↦ 0]:   {m ↦ 10, n ↦ 3, r ↦ 10}
σ[r ↦ σm, d ↦ 0]: {m ↦ 10, n ↦ 3, r ↦ 10}


## Exercise 11 — describing states (chapter has solution)

Walking through the four parts:

**(a)** A state where `x = 0` and `m = n = 1`. Simplest description: `{m ↦ 1, n ↦ 1}` — note we **don't** mention x because `x = 0` is the default; mentioning it would still be valid but redundant.

**(b)** Take that state, update `x` to `1`. Update notation: `{m ↦ 1, n ↦ 1}[x ↦ 1]`. Resulting state: `{m ↦ 1, n ↦ 1, x ↦ 1}`.

**(c)** We start in some state σ where we *only know* σm = σn = 1. There are other variables whose values we don't know. We update x to 1: `σ[x ↦ 1]`. We **can't** describe the result with curly-brace notation because we don't know all the non-zero variables — so we leave it as `σ[x ↦ 1]`. Then σm is still 1 (unchanged), σx is now 1, and σy = (σ[x ↦ 1])y is whatever σy was — which we don't know.

**(d)** Compute `({x ↦ 2, y ↦ 3}[x ↦ 1, z ↦ 2])[x ↦ 0, y ↦ 5, w ↦ 3]`.

First inner update: `{x ↦ 2, y ↦ 3}[x ↦ 1, z ↦ 2] = {x ↦ 1, y ↦ 3, z ↦ 2}` (x was 2, now 1; y untouched at 3; z gets 2).

Then outer update: `{x ↦ 1, y ↦ 3, z ↦ 2}[x ↦ 0, y ↦ 5, w ↦ 3]`. x → 0, y → 5, z untouched, w → 3. Result: `{x ↦ 0, y ↦ 5, z ↦ 2, w ↦ 3}`.

Since x = 0 is the default, the simplest description is `{y ↦ 5, z ↦ 2, w ↦ 3}`.

In [4]:
# Verify Exercise 11(d) computationally.
step1 = {**{'x': 2, 'y': 3}, 'x': 1, 'z': 2}
step2 = {**step1, 'x': 0, 'y': 5, 'w': 3}
print('After both updates:', state_to_str(step2))    # x is 0 so it's omitted in the canonical form

After both updates: {w ↦ 3, y ↦ 5, z ↦ 2}


## §3. Functions `A` and `B`

Recursive definitions over the structure of expressions. From the chapter §2.3.1 and §2.3.2:

### A : AExp × St → ℤ

$$\begin{aligned}
\mathcal{A}\,n\,\sigma &= \mathcal{N}\,n  \quad &&\text{(numeral interpretation)} \\
\mathcal{A}\,x\,\sigma &= \sigma(x) \\
\mathcal{A}(a + a')\,\sigma &= \mathcal{A}\,a\,\sigma + \mathcal{A}\,a'\,\sigma \\
\mathcal{A}(a - a')\,\sigma &= \mathcal{A}\,a\,\sigma - \mathcal{A}\,a'\,\sigma \\
\mathcal{A}(a \times a')\,\sigma &= \mathcal{A}\,a\,\sigma \cdot \mathcal{A}\,a'\,\sigma
\end{aligned}$$

### B : BExp × St → 𝔹

$$\begin{aligned}
\mathcal{B}\,\mathbf{ff}\,\sigma &= \mathbf{ff} \\
\mathcal{B}\,\mathbf{tt}\,\sigma &= \mathbf{tt} \\
\mathcal{B}(a = a')\,\sigma &= \mathbf{tt} \text{ if } \mathcal{A}\,a\,\sigma = \mathcal{A}\,a'\,\sigma,\ \mathbf{ff}\text{ otherwise} \\
\mathcal{B}(a \le a')\,\sigma &= \mathbf{tt} \text{ if } \mathcal{A}\,a\,\sigma \le \mathcal{A}\,a'\,\sigma,\ \mathbf{ff}\text{ otherwise} \\
\mathcal{B}(\neg b)\,\sigma &= \mathbf{tt} \text{ if } \mathcal{B}\,b\,\sigma = \mathbf{ff},\ \mathbf{ff}\text{ otherwise} \\
\mathcal{B}(b \wedge b')\,\sigma &= \mathbf{tt} \text{ if both } \mathcal{B}\,b\,\sigma = \mathbf{tt} \text{ and } \mathcal{B}\,b'\,\sigma = \mathbf{tt},\ \mathbf{ff}\text{ otherwise}
\end{aligned}$$

**Both are total functions** — given any expression and any state, you get a value. There's no "undefined" or "error." This is what makes the small-step rules deterministic.

## Exercise 12 — verifying Exercise 1 with `B`

> Take your solutions to Exercise 1 and show that the interpretation of the boolean expressions you give have the expected evaluation.

**Exercise 1(a):** `a ∨ b` was encoded as `¬(¬a ∧ ¬b)`. We need to show that `B(¬(¬a ∧ ¬b), σ) = tt` exactly when `B(a, σ) = tt` or `B(b, σ) = tt`.

Walk through the rules for `B`:

- `B(¬(¬a ∧ ¬b), σ) = tt` iff `B(¬a ∧ ¬b, σ) = ff` (rule for ¬).
- `B(¬a ∧ ¬b, σ) = ff` iff *not both* `B(¬a, σ) = tt` and `B(¬b, σ) = tt` (rule for ∧, contrapositive).
- That's iff `B(¬a, σ) = ff` or `B(¬b, σ) = ff`.
- That's iff `B(a, σ) = tt` or `B(b, σ) = tt` (rule for ¬, twice).

Which is exactly the meaning of `a ∨ b`. ✓

**Exercise 1(b):** `a → b` was encoded as `¬(a ∧ ¬b)`. Show `B(¬(a ∧ ¬b), σ) = tt` iff `B(a, σ) = ff` or `B(b, σ) = tt`.

- `B(¬(a ∧ ¬b), σ) = tt` iff `B(a ∧ ¬b, σ) = ff`.
- That's iff `B(a, σ) = ff` or `B(¬b, σ) = ff`.
- That's iff `B(a, σ) = ff` or `B(b, σ) = tt`. ✓

Which matches the truth table of implication.

We already did the empirical truth-table check in N2. The argument above is the *formal* version asked for in the exercise.

## §4. The small-step transition relation `⇒`

From chapter §2.3.3, here are the 6 rules. (Plus an implicit *no rule* for `skip` alone — `skip` is terminal.)

### Assignment

$$\langle x := a,\ \sigma\rangle \Rightarrow \langle \textbf{skip},\ \sigma[x \mapsto \mathcal{A}\,a\,\sigma]\rangle$$

Replace `x := a` with `skip`, and update σ to map x to the value of a.

### Sequential composition (general case)

$$\frac{\langle S,\ \sigma\rangle \Rightarrow \langle S',\ \sigma'\rangle}{\langle S; T,\ \sigma\rangle \Rightarrow \langle S'; T,\ \sigma'\rangle}$$

If `S` can take a step, then so can `S; T` — by stepping `S` and dragging `T` along unchanged.

### Sequential composition (skip case)

$$\langle \textbf{skip}; T,\ \sigma\rangle \Rightarrow \langle T,\ \sigma\rangle$$

If the left side has been reduced to `skip`, drop it and continue with `T`.

### Conditional

$$\langle \textbf{if } b \textbf{ then } S \textbf{ else }(S'),\ \sigma\rangle \Rightarrow \begin{cases}\langle S,\ \sigma\rangle & \text{if } \mathcal{B}\,b\,\sigma = \mathbf{tt} \\ \langle S',\ \sigma\rangle & \text{otherwise}\end{cases}$$

Branch on the boolean condition. Note: σ is unchanged — checking a condition doesn't change state.

### While loop

$$\langle \textbf{while } b \textbf{ do }(S),\ \sigma\rangle \Rightarrow \begin{cases}\langle S; \textbf{while } b \textbf{ do }(S),\ \sigma\rangle & \text{if } \mathcal{B}\,b\,\sigma = \mathbf{tt} \\ \langle \textbf{skip},\ \sigma\rangle & \text{otherwise}\end{cases}$$

If the loop condition is true, *unfold* the loop into the body followed by the loop again. If false, replace the whole loop with `skip`.

### Crucial property — determinism

Given `⟨S, σ⟩` with S ≠ skip, **exactly one** of these rules applies. The next configuration `⟨S', σ'⟩` is uniquely determined. So a program either terminates in a unique final state, or it doesn't terminate at all — there's no "choice."

## §5. Example 7 — formal stepwise computation

The chapter walks Example 1's division program through the small-step rules. Let's reproduce it with the interpreter.

Starting state: `σ = {m ↦ 10, n ↦ 3}`. The interpreter will produce exactly the trace the chapter shows in §2.4.

In [5]:
div_prog = '''
    r := m;
    d := 0;
    while n <= r do (
        d := d + 1;
        r := r - n
    )
'''

print(trace(div_prog, {'m': 10, 'n': 3}, view='formal'))

Where:
  L1 := while n ≤ r do (d := d + 1; r := r − n)

⟨r := m; d := 0; L1, {m ↦ 10, n ↦ 3}⟩
  ⇒  ⟨skip; d := 0; L1, {m ↦ 10, n ↦ 3, r ↦ 10}⟩    [:=]
  ⇒  ⟨d := 0; L1, {m ↦ 10, n ↦ 3, r ↦ 10}⟩    [skip-;]
  ⇒  ⟨skip; L1, {m ↦ 10, n ↦ 3, r ↦ 10}⟩    [:=]
  ⇒  ⟨L1, {m ↦ 10, n ↦ 3, r ↦ 10}⟩    [skip-;]
  ⇒  ⟨d := d + 1; r := r − n; L1, {m ↦ 10, n ↦ 3, r ↦ 10}⟩    [while-tt]
  ⇒  ⟨skip; r := r − n; L1, {d ↦ 1, m ↦ 10, n ↦ 3, r ↦ 10}⟩    [:=]
  ⇒  ⟨r := r − n; L1, {d ↦ 1, m ↦ 10, n ↦ 3, r ↦ 10}⟩    [skip-;]
  ⇒  ⟨skip; L1, {d ↦ 1, m ↦ 10, n ↦ 3, r ↦ 7}⟩    [:=]
  ⇒  ⟨L1, {d ↦ 1, m ↦ 10, n ↦ 3, r ↦ 7}⟩    [skip-;]
  ⇒  ⟨d := d + 1; r := r − n; L1, {d ↦ 1, m ↦ 10, n ↦ 3, r ↦ 7}⟩    [while-tt]
  ⇒  ⟨skip; r := r − n; L1, {d ↦ 2, m ↦ 10, n ↦ 3, r ↦ 7}⟩    [:=]
  ⇒  ⟨r := r − n; L1, {d ↦ 2, m ↦ 10, n ↦ 3, r ↦ 7}⟩    [skip-;]
  ⇒  ⟨skip; L1, {d ↦ 2, m ↦ 10, n ↦ 3, r ↦ 4}⟩    [:=]
  ⇒  ⟨L1, {d ↦ 2, m ↦ 10, n ↦ 3, r ↦ 4}⟩    [skip-;]
  ⇒  ⟨d := d + 1; r := r − n; L1, {d ↦ 2, m ↦ 10, n ↦ 3, r ↦ 4}⟩

**Compare this with Example 7 in the chapter.** A few things to notice:

1. The chapter abbreviates the body of the loop as `B` for compactness. Our interpreter does the same — it spotted the long `while` body and aliased the whole loop as `L1`. Same convention.
2. The chapter uses `⇒⁴` notation to mean "4 steps in one go" when several assignments fire in sequence. We expand each step individually — easier to follow when learning.
3. Each transition is labelled with the rule that fired (`:=`, `;`, `skip-;`, `while-tt`, `while-ff`).
4. The final config is `⟨skip, {d ↦ 3, m ↦ 10, n ↦ 3, r ↦ 1}⟩` — exactly the chapter's predicted final state.

## Example 8 — state-tracking table

When the chapter asks you to "track the execution" of a program (Example 8 and Exercise 13), the expected output is a compact table — variables across the top, one row per significant transition. Variables that don't change are typically omitted.

Our interpreter's `view='table'` produces this:

In [6]:
print(trace(div_prog, {'m': 10, 'n': 3}, view='table'))

step | rule     | d | r 
-----+----------+---+---
0    | start    | 0 | 0 
1    | :=       | 0 | 10
2    | skip-;   | 0 | 10
3    | :=       | 0 | 10
4    | skip-;   | 0 | 10
5    | while-tt | 0 | 10
6    | :=       | 1 | 10
7    | skip-;   | 1 | 10
8    | :=       | 1 | 7 
9    | skip-;   | 1 | 7 
10   | while-tt | 1 | 7 
11   | :=       | 2 | 7 
12   | skip-;   | 2 | 7 
13   | :=       | 2 | 4 
14   | skip-;   | 2 | 4 
15   | while-tt | 2 | 4 
16   | :=       | 3 | 4 
17   | skip-;   | 3 | 4 
18   | :=       | 3 | 1 
19   | skip-;   | 3 | 1 
20   | while-ff | 3 | 1 


Compare with the chapter's table for Example 8:

```
        d   r
start   0   10
loop    1   7
loop    2   4
loop    3   1
```

The chapter shows only one row per loop iteration. Our interpreter is more verbose — it shows every transition, including the `;`, `skip-;`, and `while-tt` rule firings between each loop body. For exam-style answers you'd compress those into a single row per iteration.

Both contain the same information. For learning, the verbose version is more honest about what's happening. For the exam, you'd use the compact version.

## Exercise 13 — full transitions of gcd(12, 30)

> For the program determining the greatest common divisor `gcd` of two inputs held in `m` and `n` from Example 3, compute all its transitions in the small-step semantics for a state σ with σm = 12 and σn = 30. Make sure to justify each step.

The gcd program (Example 3):

In [7]:
gcd_prog = '''
    x := m; y := n;
    while !(x = y) do (
        if !(x <= y) then
            x := x - y
        else (
            y := y - x
        )
    )
'''

# Full formal trace for σ = {m ↦ 12, n ↦ 30}.
print(trace(gcd_prog, {'m': 12, 'n': 30}, view='formal'))

Where:
  L1 := while ¬x = y do (if ¬x ≤ y then x := x − y else (y := y − x))

⟨x := m; y := n; L1, {m ↦ 12, n ↦ 30}⟩
  ⇒  ⟨skip; y := n; L1, {m ↦ 12, n ↦ 30, x ↦ 12}⟩    [:=]
  ⇒  ⟨y := n; L1, {m ↦ 12, n ↦ 30, x ↦ 12}⟩    [skip-;]
  ⇒  ⟨skip; L1, {m ↦ 12, n ↦ 30, x ↦ 12, y ↦ 30}⟩    [:=]
  ⇒  ⟨L1, {m ↦ 12, n ↦ 30, x ↦ 12, y ↦ 30}⟩    [skip-;]
  ⇒  ⟨if ¬x ≤ y then x := x − y else (y := y − x); L1, {m ↦ 12, n ↦ 30, x ↦ 12, y ↦ 30}⟩    [while-tt]
  ⇒  ⟨y := y − x; L1, {m ↦ 12, n ↦ 30, x ↦ 12, y ↦ 30}⟩    [if-ff]
  ⇒  ⟨skip; L1, {m ↦ 12, n ↦ 30, x ↦ 12, y ↦ 18}⟩    [:=]
  ⇒  ⟨L1, {m ↦ 12, n ↦ 30, x ↦ 12, y ↦ 18}⟩    [skip-;]
  ⇒  ⟨if ¬x ≤ y then x := x − y else (y := y − x); L1, {m ↦ 12, n ↦ 30, x ↦ 12, y ↦ 18}⟩    [while-tt]
  ⇒  ⟨y := y − x; L1, {m ↦ 12, n ↦ 30, x ↦ 12, y ↦ 18}⟩    [if-ff]
  ⇒  ⟨skip; L1, {m ↦ 12, n ↦ 30, x ↦ 12, y ↦ 6}⟩    [:=]
  ⇒  ⟨L1, {m ↦ 12, n ↦ 30, x ↦ 12, y ↦ 6}⟩    [skip-;]
  ⇒  ⟨if ¬x ≤ y then x := x − y else (y := y − x); L1, {m ↦ 12, n ↦ 30, x ↦ 12, y ↦ 6}⟩  

Each transition is justified by the rule shown in `[brackets]` at the end of the line:

- `:=` — assignment rule
- `;` — general sequential composition rule (a step inside the left of a `;`)
- `skip-;` — drop a `skip` from the left of a `;`
- `if-tt` / `if-ff` — conditional, depending on `B(b, σ)`
- `while-tt` / `while-ff` — loop, depending on `B(b, σ)`

The final state shows `x = y = 6`, the gcd of 12 and 30. ✓

### Compact table version (more like exam style)

If asked to *track* the execution rather than give every transition, the table is shorter:

In [8]:
print(trace(gcd_prog, {'m': 12, 'n': 30}, view='table'))

step | rule     | x  | y 
-----+----------+----+---
0    | start    | 0  | 0 
1    | :=       | 12 | 0 
2    | skip-;   | 12 | 0 
3    | :=       | 12 | 30
4    | skip-;   | 12 | 30
5    | while-tt | 12 | 30
6    | if-ff    | 12 | 30
7    | :=       | 12 | 18
8    | skip-;   | 12 | 18
9    | while-tt | 12 | 18
10   | if-ff    | 12 | 18
11   | :=       | 12 | 6 
12   | skip-;   | 12 | 6 
13   | while-tt | 12 | 6 
14   | if-tt    | 12 | 6 
15   | :=       | 6  | 6 
16   | skip-;   | 6  | 6 
17   | while-ff | 6  | 6 


## §6. Non-termination — Examples 9 and 10

The chapter notes that some programs *don't terminate* — there's no final `⟨skip, σ'⟩` configuration to reach.

### Example 9 — the obvious non-terminator

```
x := 0;
while 0 <= x do (x := x + 1)
```

x stays non-negative forever. Each iteration: check `0 ≤ x` (true), unfold loop, run body (`x := x + 1`), administrative `;` and `skip-;` rules, back to the loop. Forever.

Our interpreter detects this with the `max_steps` budget:

In [9]:
ex9 = '''
    x := 0;
    while 0 <= x do (
        x := x + 1
    )
'''

# Cap at 20 steps so the trace is readable. The truncation marker tells us non-termination.
print(trace(ex9, {}, view='formal', max_steps=20))

Where:
  L1 := while 0 ≤ x do (x := x + 1)

⟨x := 0; L1, {}⟩
  ⇒  ⟨skip; L1, {}⟩    [:=]
  ⇒  ⟨L1, {}⟩    [skip-;]
  ⇒  ⟨x := x + 1; L1, {}⟩    [while-tt]
  ⇒  ⟨skip; L1, {x ↦ 1}⟩    [:=]
  ⇒  ⟨L1, {x ↦ 1}⟩    [skip-;]
  ⇒  ⟨x := x + 1; L1, {x ↦ 1}⟩    [while-tt]
  ⇒  ⟨skip; L1, {x ↦ 2}⟩    [:=]
  ⇒  ⟨L1, {x ↦ 2}⟩    [skip-;]
  ⇒  ⟨x := x + 1; L1, {x ↦ 2}⟩    [while-tt]
  ⇒  ⟨skip; L1, {x ↦ 3}⟩    [:=]
  ⇒  ⟨L1, {x ↦ 3}⟩    [skip-;]
  ⇒  ⟨x := x + 1; L1, {x ↦ 3}⟩    [while-tt]
  ⇒  ⟨skip; L1, {x ↦ 4}⟩    [:=]
  ⇒  ⟨L1, {x ↦ 4}⟩    [skip-;]
  ⇒  ⟨x := x + 1; L1, {x ↦ 4}⟩    [while-tt]
  ⇒  ⟨skip; L1, {x ↦ 5}⟩    [:=]
  ⇒  ⟨L1, {x ↦ 5}⟩    [skip-;]
  ⇒  ⟨x := x + 1; L1, {x ↦ 5}⟩    [while-tt]
  ⇒  ⟨skip; L1, {x ↦ 6}⟩    [:=]
  ⇒  ⟨L1, {x ↦ 6}⟩    [skip-;]
  ⇒  ... step budget exceeded — likely non-terminating ...


### Example 10 — the *non-obvious* non-terminator

Recall Example 2: search for an integer solution to `ax² + bx + c = 0`. For `a = 1, b = -2, c = 2` the equation `x² − 2x + 2 = 0` has no real roots (discriminant is negative). So the search runs forever.

But unlike Example 9, this isn't *obvious* from the program text. You have to reason about the arithmetic to know it doesn't terminate.

In [10]:
quad_prog = '''
    x := 0; y := 0;
    while !(y = 1) do (
        if a * x * x + b * x + c = 0 then
            y := 1
        else (
            if a * x * x - b * x + c = 0 then
                y := 1
            else (
                skip
            )
        );
        x := x + 1
    )
'''

# x² − 2x + 2 has no real roots → non-terminating.
print(trace(quad_prog, {'a': 1, 'b': -2, 'c': 2}, view='table', max_steps=80))

step | rule     | x 
-----+----------+---
0    | start    | 0 
1    | :=       | 0 
2    | skip-;   | 0 
3    | :=       | 0 
4    | skip-;   | 0 
5    | while-tt | 0 
6    | if-ff    | 0 
7    | if-ff    | 0 
8    | skip-;   | 0 
9    | :=       | 1 
10   | skip-;   | 1 
11   | while-tt | 1 
12   | if-ff    | 1 
13   | if-ff    | 1 
14   | skip-;   | 1 
15   | :=       | 2 
16   | skip-;   | 2 
17   | while-tt | 2 
18   | if-ff    | 2 
19   | if-ff    | 2 
20   | skip-;   | 2 
21   | :=       | 3 
22   | skip-;   | 3 
23   | while-tt | 3 
24   | if-ff    | 3 
25   | if-ff    | 3 
26   | skip-;   | 3 
27   | :=       | 4 
28   | skip-;   | 4 
29   | while-tt | 4 
30   | if-ff    | 4 
31   | if-ff    | 4 
32   | skip-;   | 4 
33   | :=       | 5 
34   | skip-;   | 5 
35   | while-tt | 5 
36   | if-ff    | 5 
37   | if-ff    | 5 
38   | skip-;   | 5 
39   | :=       | 6 
40   | skip-;   | 6 
41   | while-tt | 6 
42   | if-ff    | 6 
43   | if-ff    | 6 
44   | skip-;   | 6 
45   | :=    

**The chapter's argument** for non-termination of `x² − 2x + 2 = 0`:

- For all `x ∈ ℕ`, the expression `x² + 2x + 2` is positive (sum of non-negative terms plus 2).
- For `x ≥ 2`, we have `x² ≥ 2x`, so `x² − 2x + 2 ≥ 2 > 0`.
- For `x = 0`: `0 − 0 + 2 = 2 > 0`. For `x = 1`: `1 − 2 + 2 = 1 > 0`.
- So neither expression ever hits 0 for any `x ∈ ℕ`. The loop's exit condition `y = 1` never gets satisfied. The program runs forever.

**This kind of argument matters for the exam.** Knowing a program doesn't terminate sometimes requires *reasoning about its arithmetic*, not just inspecting its control flow.

## §7. Example 11 — gcd of 48 and 210

The chapter shows the table for the alternative gcd program (Example 3 again, called from N=48, m=210). Same algorithm, larger inputs.

In [11]:
# We use the alternative formulation that operates on m and n directly,
# matching the chapter's Example 11 program.
gcd_alt = '''
    while !(m = n) do (
        if !(n <= m) then
            n := n - m
        else (
            m := m - n
        )
    )
'''

print(trace(gcd_alt, {'m': 48, 'n': 210}, view='table'))

step | rule     | m  | n  
-----+----------+----+----
0    | start    | 48 | 210
1    | while-tt | 48 | 210
2    | if-tt    | 48 | 210
3    | :=       | 48 | 162
4    | skip-;   | 48 | 162
5    | while-tt | 48 | 162
6    | if-tt    | 48 | 162
7    | :=       | 48 | 114
8    | skip-;   | 48 | 114
9    | while-tt | 48 | 114
10   | if-tt    | 48 | 114
11   | :=       | 48 | 66 
12   | skip-;   | 48 | 66 
13   | while-tt | 48 | 66 
14   | if-tt    | 48 | 66 
15   | :=       | 48 | 18 
16   | skip-;   | 48 | 18 
17   | while-tt | 48 | 18 
18   | if-ff    | 48 | 18 
19   | :=       | 30 | 18 
20   | skip-;   | 30 | 18 
21   | while-tt | 30 | 18 
22   | if-ff    | 30 | 18 
23   | :=       | 12 | 18 
24   | skip-;   | 12 | 18 
25   | while-tt | 12 | 18 
26   | if-tt    | 12 | 18 
27   | :=       | 12 | 6  
28   | skip-;   | 12 | 6  
29   | while-tt | 12 | 6  
30   | if-ff    | 12 | 6  
31   | :=       | 6  | 6  
32   | skip-;   | 6  | 6  
33   | while-ff | 6  | 6  


## Exercise 14 — formal trace of the Diophantine program (chapter has solution)

**This is the showpiece exercise.** Take the Exercise 7 solution (the `mx + ny = k` searcher) and produce its complete small-step trace from σ = {m ↦ 3, n ↦ 4, k ↦ 7} until reaching `⟨skip, σ'⟩`.

We've already verified our Exercise 7 program in N2 — it returns `(x=1, y=1)`. Here's the full formal derivation.

In [12]:
diophantine = '''
    x := 0; y := 0;
    while !(m * x + n * y = k) do (
        x := x + 1;
        y := 0;
        while !(m * x + n * y = k) & y <= x & !(y = x) do (
            if m * y + n * x = k then (
                z := x;
                x := y;
                y := z
            ) else (
                y := y + 1
            )
        )
    )
'''

print(trace(diophantine, {'m': 3, 'n': 4, 'k': 7}, view='formal'))

Where:
  L1 := while ¬(m × x) + (n × y) = k do (x := x + 1; y := 0; while (¬(m × x) + (n × y) = k ∧ y ≤ x) ∧ ¬y = x do (if (m × y) + (n × x) = k then z := x; x := y; y := z else (y := y + 1)))
  L2 := while (¬(m × x) + (n × y) = k ∧ y ≤ x) ∧ ¬y = x do (if (m × y) + (n × x) = k then z := x; x := y; y := z else (y := y + 1))

⟨x := 0; y := 0; L1, {k ↦ 7, m ↦ 3, n ↦ 4}⟩
  ⇒  ⟨skip; y := 0; L1, {k ↦ 7, m ↦ 3, n ↦ 4}⟩    [:=]
  ⇒  ⟨y := 0; L1, {k ↦ 7, m ↦ 3, n ↦ 4}⟩    [skip-;]
  ⇒  ⟨skip; L1, {k ↦ 7, m ↦ 3, n ↦ 4}⟩    [:=]
  ⇒  ⟨L1, {k ↦ 7, m ↦ 3, n ↦ 4}⟩    [skip-;]
  ⇒  ⟨x := x + 1; y := 0; L2; L1, {k ↦ 7, m ↦ 3, n ↦ 4}⟩    [while-tt]
  ⇒  ⟨skip; y := 0; L2; L1, {k ↦ 7, m ↦ 3, n ↦ 4, x ↦ 1}⟩    [:=]
  ⇒  ⟨y := 0; L2; L1, {k ↦ 7, m ↦ 3, n ↦ 4, x ↦ 1}⟩    [skip-;]
  ⇒  ⟨skip; L2; L1, {k ↦ 7, m ↦ 3, n ↦ 4, x ↦ 1}⟩    [:=]
  ⇒  ⟨L2; L1, {k ↦ 7, m ↦ 3, n ↦ 4, x ↦ 1}⟩    [skip-;]
  ⇒  ⟨if (m × y) + (n × x) = k then z := x; x := y; y := z else (y := y + 1); L2; L1, {k ↦ 7, m ↦ 3, n ↦ 4, x ↦ 1}⟩

**Compare with the chapter's solution** in `exersise14_solution.txt`. The structure is the same — outer loop iterations advance `x`, inner loop iterations advance `y`, and the `if` checks both `mx + ny = k` and the swapped form `my + nx = k`.

The final state has `m = 3, n = 4, k = 7, x = 1, y = 1` — confirming `3·1 + 4·1 = 7`. ✓

### Exercise 14 part (b) — two more states to try

**(i)** σ = {m ↦ -6, n ↦ 8, k ↦ 4}. The chapter shows by table that this terminates with x = 2, y = 2: `−6·2 + 8·2 = −12 + 16 = 4`. ✓

In [13]:
print(trace(diophantine, {'m': -6, 'n': 8, 'k': 4}, view='table'))

step | rule     | x | y
-----+----------+---+--
0    | start    | 0 | 0
1    | :=       | 0 | 0
2    | skip-;   | 0 | 0
3    | :=       | 0 | 0
4    | skip-;   | 0 | 0
5    | while-tt | 0 | 0
6    | :=       | 1 | 0
7    | skip-;   | 1 | 0
8    | :=       | 1 | 0
9    | skip-;   | 1 | 0
10   | while-tt | 1 | 0
11   | if-ff    | 1 | 0
12   | :=       | 1 | 1
13   | skip-;   | 1 | 1
14   | while-ff | 1 | 1
15   | skip-;   | 1 | 1
16   | while-tt | 1 | 1
17   | :=       | 2 | 1
18   | skip-;   | 2 | 1
19   | :=       | 2 | 0
20   | skip-;   | 2 | 0
21   | while-tt | 2 | 0
22   | if-ff    | 2 | 0
23   | :=       | 2 | 1
24   | skip-;   | 2 | 1
25   | while-tt | 2 | 1
26   | if-ff    | 2 | 1
27   | :=       | 2 | 2
28   | skip-;   | 2 | 2
29   | while-ff | 2 | 2
30   | skip-;   | 2 | 2
31   | while-ff | 2 | 2


**(ii)** σ = {m ↦ -6, n ↦ 9, k ↦ 2}. Chapter argues by **divisibility**: both `−6` and `9` are multiples of 3, so any combination `−6x + 9y` is also a multiple of 3. But `2` isn't a multiple of 3 — so the equation has no integer solution. The program runs forever.

In [14]:
# Cap at 100 steps to demonstrate non-termination without producing thousands of lines.
print(trace(diophantine, {'m': -6, 'n': 9, 'k': 2}, view='table', max_steps=100))

step | rule     | x | y
-----+----------+---+--
0    | start    | 0 | 0
1    | :=       | 0 | 0
2    | skip-;   | 0 | 0
3    | :=       | 0 | 0
4    | skip-;   | 0 | 0
5    | while-tt | 0 | 0
6    | :=       | 1 | 0
7    | skip-;   | 1 | 0
8    | :=       | 1 | 0
9    | skip-;   | 1 | 0
10   | while-tt | 1 | 0
11   | if-ff    | 1 | 0
12   | :=       | 1 | 1
13   | skip-;   | 1 | 1
14   | while-ff | 1 | 1
15   | skip-;   | 1 | 1
16   | while-tt | 1 | 1
17   | :=       | 2 | 1
18   | skip-;   | 2 | 1
19   | :=       | 2 | 0
20   | skip-;   | 2 | 0
21   | while-tt | 2 | 0
22   | if-ff    | 2 | 0
23   | :=       | 2 | 1
24   | skip-;   | 2 | 1
25   | while-tt | 2 | 1
26   | if-ff    | 2 | 1
27   | :=       | 2 | 2
28   | skip-;   | 2 | 2
29   | while-ff | 2 | 2
30   | skip-;   | 2 | 2
31   | while-tt | 2 | 2
32   | :=       | 3 | 2
33   | skip-;   | 3 | 2
34   | :=       | 3 | 0
35   | skip-;   | 3 | 0
36   | while-tt | 3 | 0
37   | if-ff    | 3 | 0
38   | :=       | 3 | 1
39   | skip-;   

## Exercise 15 — argue Exercise 4's encoding is faithful

> Argue that your solution to Exercise 4 behaves in the same way as the original `if` statement using the small-step semantics.

**The claim:** for any boolean `b`, statements `S` and `S'`, and start state σ, the program `if b then S else (S')` and its no-if encoding produce the same final state (or both fail to terminate).

Recall the no-if encoding:

```
done := 0;
while b & done = 0 do (S; done := 1);
while !(done = 1) & !b do (S'; done := 1)
```

Where `done` is a fresh variable not mentioned in `S`, `S'`, or `b`.

### Argument (informal but rigorous)

**Case 1: `B(b, σ) = tt`.**

Original: `⟨if b then S else S', σ⟩` ⇒ `⟨S, σ⟩` by `if-tt`. From here, S runs to its conclusion (or fails to terminate). Final state: whatever S produces.

No-if: 
1. `done := 0` runs, σ becomes σ[done ↦ 0]. Call this σ₀.
2. First while: condition is `b ∧ done = 0`. In σ₀, `B(b, σ₀) = tt` (b doesn't mention done, so its value in σ₀ matches its value in σ). And `done = 0` is true. So `B(b ∧ done = 0, σ₀) = tt`. Loop unfolds.
3. Body of first while: `S; done := 1`. Run S in σ₀ — produces some σ₁ (with the same final state as the original case 1, modulo done = 0). Then `done := 1`, giving σ₁[done ↦ 1]. Call this τ.
4. First while iterates: condition is `b ∧ done = 0`. In τ, `done = 1` makes the AND false. Loop exits with state τ.
5. Second while: condition is `¬(done = 1) ∧ ¬b`. In τ, `done = 1`, so `¬(done = 1) = ff`, AND is false. Loop body doesn't fire. Final state: τ.

τ differs from "original case 1's final state" only by having `done = 1`. Since `done` is fresh, no observable difference. ✓

**Case 2: `B(b, σ) = ff`.**

Original: `⟨if b then S else S', σ⟩` ⇒ `⟨S', σ⟩` by `if-ff`. S' runs to its conclusion.

No-if:
1. `done := 0` runs, giving σ₀ = σ[done ↦ 0].
2. First while: condition `b ∧ done = 0`. `B(b, σ₀) = ff` makes the AND false. Loop body doesn't fire. State stays σ₀.
3. Second while: condition `¬(done = 1) ∧ ¬b`. In σ₀, `done = 0` so `¬(done = 1) = tt`. Also `¬b = tt`. AND is true. Loop unfolds.
4. Body: `S'; done := 1`. Run S' in σ₀ — produces some σ₁. Then `done := 1`, giving τ = σ₁[done ↦ 1].
5. Second while iterates: in τ, `done = 1` so `¬(done = 1) = ff`. Loop exits.

τ matches "original case 2's final state" up to the value of `done`. ✓

**Termination matches:** the no-if encoding only ever runs S or S' once (the flag ensures this). It terminates iff the original does.

**The chapter notes this argument is "not straightforward."** A fully rigorous version would do structural induction over derivations and explicitly state "done is fresh in S, S', and b." The argument above captures the operational reasoning.

### Empirical sanity check

Let's verify on concrete examples:

In [15]:
# For every boolean b that's true or false in σ, original and no-if should produce same state
# (modulo the `done` variable which is internal to the encoding).

original_if = 'if x = 0 then (y := 100) else (y := 200)'
no_if = '''
    done := 0;
    while x = 0 & done = 0 do (
        y := 100;
        done := 1
    );
    while !(done = 1) & !(x = 0) do (
        y := 200;
        done := 1
    )
'''

for x in [0, 5, -3, 100]:
    sigma = {'x': x}
    orig = run(original_if, sigma)
    enc = run(no_if, sigma)
    enc_minus_done = {k: v for k, v in enc.items() if k != 'done'}
    print(f'x = {x:4}:  original = {orig},  no-if (sans done) = {enc_minus_done},  match = {orig == enc_minus_done}')

x =    0:  original = {'y': 100},  no-if (sans done) = {'y': 100},  match = True
x =    5:  original = {'x': 5, 'y': 200},  no-if (sans done) = {'x': 5, 'y': 200},  match = True
x =   -3:  original = {'x': -3, 'y': 200},  no-if (sans done) = {'x': -3, 'y': 200},  match = True
x =  100:  original = {'x': 100, 'y': 200},  no-if (sans done) = {'x': 100, 'y': 200},  match = True


## Exercise 16 — small-step rule for `repeat S until b` (chapter has solution)

> Imagine the grammar of While was changed to include `repeat S until b` rather than while loops. What should the rule for a stepwise operational semantics look like?

### Naïve (wrong) attempt:

$$\langle \textbf{repeat } S \textbf{ until } b,\ \sigma\rangle \Rightarrow_R \langle S; \textbf{repeat } S \textbf{ until } b,\ \sigma\rangle$$

This is wrong! It would just keep unfolding `S` forever, never checking `b`.

### Correct rule (chapter's solution):

$$\langle \textbf{repeat } S \textbf{ until } b,\ \sigma\rangle \Rightarrow_R \langle S; \textbf{if } b \textbf{ then skip else (repeat } S \textbf{ until } b\textbf{)},\ \sigma\rangle$$

After running `S` once, **check `b`**: if true, we're done (`skip`); if false, repeat.

### What you'd need to prove

Recall from Exercise 3 that `repeat S until b` was supposed to be expressible as `S; while ¬b do S`. To show our `⇒_R` rule "captures the expected behavior," you'd prove:

$$\langle \textbf{repeat } S \textbf{ until } b,\ \sigma\rangle \Rightarrow_R^* \langle \textbf{skip},\ \sigma'\rangle \quad \text{iff} \quad \langle S; \textbf{while } \neg b \textbf{ do } S,\ \sigma\rangle \Rightarrow^* \langle \textbf{skip},\ \sigma'\rangle$$

The chapter notes this is hard with small-step semantics. The big-step semantics in the appendix makes it easier — but we don't cover that.

### Why it's hard with small-step

The two systems take different numbers of steps and pass through different intermediate configurations. Step-by-step equivalence doesn't hold. What you need is **trace equivalence** at the endpoints: same start state ⇒ same final state (or both diverge). Proving that requires structural induction over the rules of `⇒_R` and `⇒`, which gets verbose.

We sketch the case structure but won't carry out the full induction here.

## Summary

What you've now seen:

- The formal definition of a **state** σ — a function from variables to integers, finitely many non-zero.
- The state update notation `σ[x ↦ n]` — both single and multi-variable.
- Functions `A` and `B` for evaluating expressions in a given state.
- The 6 inference rules of the small-step semantics, applied to all chapter examples.
- Examples 7–11 reproduced as live interpreter runs.
- Exercise 13's full transition trace for gcd(12, 30).
- Exercise 14's showpiece — formal trace of the Diophantine searcher, including non-terminating cases.
- Exercise 15's correctness argument for the no-if encoding.
- Exercise 16's correct rule for `repeat-until`.

Next: **N4 — reasoning about the semantics**. We tackle non-termination *as a property*, Propositions 2 and 3 (associativity and structural decomposition of `;`), exercises 17 and 18 (proof-style — flagged as best-effort), the quiz, and the closing "translate Python intuition into formal exam answers" section.